## CLIP-RGB Baseline on EuroSAT-MS

**Protocol: Image-to-Image Retrieval** (val → query, test → gallery)

# Objectives
- Build the EuroSAT-MS query (val) and gallery (test) splits.
- Use only RGB bands **B04, B03, B02** as CLIP input.
- Run a frozen **CLIP ViT-B/16** image encoder.
- Measure **image-to-image retrieval** with cosine similarity.
- Each query *image* is matched against all gallery *images*; a gallery item is relevant iff it shares the same class label.
- Report **Recall@1, Recall@5, Recall@10, mAP**.


## Setup project paths and imports

In [1]:
import sys
import random
from pathlib import Path

import clip
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data" / "EuroSAT_MS"
CKPT_PATH = PROJECT_ROOT / "checkpoints" / "ViT-B-16.pt"

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("CKPT_PATH exists:", CKPT_PATH.exists())


DATA_ROOT exists: True
CKPT_PATH exists: True


## Import dataset and helpers from `src/`

In [2]:
from src.datasets.eurosat import (
    build_eurosat_dataloaders,
    SENTINEL2_BANDS,
)
from src.baselines.pca_baseline import load_openai_clip_model
from src.models.clip_utils import encode_test_gallery_rgb
from src.utils.visualization import extract_rgb_bands, stretch_for_display
from src.baselines.rs_transclip_baseline import evaluate_single_label_retrieval_from_similarity


## Build query (val) and gallery (test) dataloaders

**Split:** train 80 % — query/val 10 % — gallery/test 10 %  
Query and gallery are **disjoint** image sets so a query image is never in its own gallery.

In [3]:
bundle = build_eurosat_dataloaders(
    root=DATA_ROOT,
    batch_size=64,
    num_workers=0,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
    normalize=True,
    clamp_range=(0.0, 1.0),
    transform=None,
)

dataset      = bundle["dataset"]
query_loader = bundle["loaders"]["val"]   # val → query
gallery_loader = bundle["loaders"]["test"] # test → gallery

print("Num classes  :", len(dataset.class_names))
print("Class names  :", dataset.class_names)
print("Query size   :", len(query_loader.dataset))
print("Gallery size :", len(gallery_loader.dataset))


Num classes  : 10
Class names  : ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Query size   : 2700
Gallery size : 2700


## Load CLIP model

In [4]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

# load_openai_clip_model searches CKPT_PATH first, then falls back to clip cache.
model, clip_tokenize = load_openai_clip_model(CKPT_PATH, device)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print("CLIP loaded successfully.")


Device: mps
CLIP loaded successfully.


## Encode query images and gallery images

Both are encoded with the **CLIP image encoder** using only the RGB bands (B04, B03, B02).  
Each embedding is L2-normalised (unit vector).

In [5]:
RGB_INDICES = (3, 2, 1)  # B04=index 3, B03=index 2, B02=index 1 in EuroSAT-MS

print("Encoding query images (val split) ...")
query_result = encode_test_gallery_rgb(
    query_loader,
    model,
    device=device,
    image_size=224,
    rgb_indices=RGB_INDICES,
    show_progress=True,
)

print("\nEncoding gallery images (test split) ...")
gallery_result = encode_test_gallery_rgb(
    gallery_loader,
    model,
    device=device,
    image_size=224,
    rgb_indices=RGB_INDICES,
    show_progress=True,
)

print("\nQuery  features:", query_result["features"].shape)
print("Gallery features:", gallery_result["features"].shape)


Encoding query images (val split) ...


Encoding test gallery:   0%|          | 0/43 [00:00<?, ?it/s]


Encoding gallery images (test split) ...


Encoding test gallery:   0%|          | 0/43 [00:00<?, ?it/s]


Query  features: torch.Size([2700, 512])
Gallery features: torch.Size([2700, 512])


## Run image-to-image retrieval

Compute cosine similarity matrix `[Q, N_gallery]` and rank gallery items per query.

In [6]:
similarity = query_result["features"] @ gallery_result["features"].T  # [Q, N]

metrics, ranked_indices, ranked_relevance, per_query_results = \
    evaluate_single_label_retrieval_from_similarity(
        similarity,
        query_labels=query_result["labels"],
        gallery_labels=gallery_result["labels"],
        ks=(1, 5, 10),
    )

print("=== CLIP-RGB baseline — Image-to-Image Retrieval (EuroSAT val→test) ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.2f}%")


=== CLIP-RGB baseline — Image-to-Image Retrieval (EuroSAT val→test) ===
  R@1: 78.89%
  R@5: 94.48%
  R@10: 97.37%
  mAP: 41.01%


## Per-query retrieval breakdown

In [7]:
idx_to_class = {i: name for i, name in enumerate(dataset.class_names)}

print(f"{'Class':>22} | {'AP':>6} | {'R@1':>4} | {'R@5':>4} | {'R@10':>5} | first_hit")
print("-" * 70)

# Aggregate per class
from collections import defaultdict
class_hits = defaultdict(lambda: {"AP": [], "hit1": [], "hit5": [], "hit10": [], "rank": []})

for row in per_query_results:
    c = idx_to_class[row["query_label"]]
    class_hits[c]["AP"].append(row["AP_percent"])
    class_hits[c]["hit1"].append(int(row["hit@1"]))
    class_hits[c]["hit5"].append(int(row["hit@5"]))
    class_hits[c]["hit10"].append(int(row["hit@10"]))
    class_hits[c]["rank"].append(row["first_hit_rank"] or 9999)

for cls in sorted(class_hits):
    d = class_hits[cls]
    print(
        f"{cls:>22} | "
        f"{sum(d['AP'])/len(d['AP']):5.1f}% | "
        f"{sum(d['hit1'])/len(d['hit1'])*100:4.1f} | "
        f"{sum(d['hit5'])/len(d['hit5'])*100:4.1f} | "
        f"{sum(d['hit10'])/len(d['hit10'])*100:5.1f} | "
        f"avg_rank={sum(d['rank'])/len(d['rank']):.1f}"
    )


                 Class |     AP |  R@1 |  R@5 |  R@10 | first_hit
----------------------------------------------------------------------
            AnnualCrop |  40.5% | 79.7 | 96.0 |  97.3 | avg_rank=2.2
                Forest |  71.7% | 95.0 | 99.7 | 100.0 | avg_rank=1.1
  HerbaceousVegetation |  45.2% | 86.3 | 96.7 |  98.3 | avg_rank=1.5
               Highway |  17.7% | 50.8 | 84.0 |  92.4 | avg_rank=3.6
            Industrial |  54.0% | 89.6 | 98.4 |  99.6 | avg_rank=1.3
               Pasture |  21.7% | 70.0 | 92.5 |  98.0 | avg_rank=2.3
         PermanentCrop |  23.8% | 72.4 | 92.4 |  95.6 | avg_rank=2.7
           Residential |  37.9% | 87.7 | 96.3 |  98.7 | avg_rank=1.7
                 River |  21.3% | 58.8 | 91.2 |  95.6 | avg_rank=2.8
               SeaLake |  61.9% | 88.3 | 95.0 |  97.3 | avg_rank=2.5


## Multi-seed robustness check

Each seed produces a different train/val/test split so the query and gallery sets change. Mean ± std over 5 seeds estimates result variance.

In [8]:
SEEDS = [42, 43, 44, 45, 46]

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def run_image_to_image_eval_for_seed(seed: int):
    set_all_seeds(seed)

    bnd = build_eurosat_dataloaders(
        root=DATA_ROOT,
        batch_size=64,
        num_workers=0,
        train_ratio=0.8,
        val_ratio=0.1,
        test_ratio=0.1,
        seed=seed,
        normalize=True,
        clamp_range=(0.0, 1.0),
        transform=None,
    )
    q_res = encode_test_gallery_rgb(
        bnd["loaders"]["val"], model, device=device,
        image_size=224, rgb_indices=RGB_INDICES, show_progress=False,
    )
    g_res = encode_test_gallery_rgb(
        bnd["loaders"]["test"], model, device=device,
        image_size=224, rgb_indices=RGB_INDICES, show_progress=False,
    )
    sim = q_res["features"] @ g_res["features"].T
    m, _, _, _ = evaluate_single_label_retrieval_from_similarity(
        sim,
        query_labels=q_res["labels"],
        gallery_labels=g_res["labels"],
        ks=(1, 5, 10),
    )
    return {
        "seed": seed,
        "num_queries": int(q_res["labels"].numel()),
        "num_gallery": int(g_res["labels"].numel()),
        "R@1":  float(m["R@1"]),
        "R@5":  float(m["R@5"]),
        "R@10": float(m["R@10"]),
        "mAP":  float(m["mAP"]),
    }


all_results = []
for seed in SEEDS:
    print(f"Running seed = {seed} ...")
    res = run_image_to_image_eval_for_seed(seed)
    all_results.append(res)
    print(
        f"  R@1={res['R@1']:.2f}% | R@5={res['R@5']:.2f}% | "
        f"R@10={res['R@10']:.2f}% | mAP={res['mAP']:.2f}%"
    )

results_df = pd.DataFrame(all_results)
print("\n=== Per-seed results ===")
display(results_df)

metric_cols = ["R@1", "R@5", "R@10", "mAP"]
summary_rows = []
for col in metric_cols:
    mean_val = results_df[col].mean()
    std_val  = results_df[col].std(ddof=1)
    summary_rows.append({
        "metric":      col,
        "mean_percent": round(mean_val, 2),
        "std_percent":  round(std_val,  2),
        "mean ± std":  f"{mean_val:.2f} ± {std_val:.2f}",
    })

multi_seed_summary_df = pd.DataFrame(summary_rows)
print("\n=== Mean ± std across seeds ===")
display(multi_seed_summary_df)


Running seed = 42 ...
  R@1=78.89% | R@5=94.48% | R@10=97.37% | mAP=41.01%
Running seed = 43 ...
  R@1=76.70% | R@5=94.67% | R@10=97.67% | mAP=39.58%
Running seed = 44 ...
  R@1=77.70% | R@5=93.48% | R@10=97.11% | mAP=40.08%
Running seed = 45 ...
  R@1=77.22% | R@5=94.19% | R@10=97.22% | mAP=39.60%
Running seed = 46 ...
  R@1=77.85% | R@5=94.00% | R@10=97.22% | mAP=41.33%

=== Per-seed results ===


,seed,num_queries,num_gallery,R@1,R@5,R@10,mAP
0,42,2700,2700,78.888887,94.481480,97.370368,41.005860
1,43,2700,2700,76.703703,94.666666,97.666669,39.581842
2,44,2700,2700,77.703702,93.481481,97.111112,40.075958
3,45,2700,2700,77.222222,94.185185,97.222221,39.601995
4,46,2700,2700,77.851850,94.000000,97.222221,41.334358



=== Mean ± std across seeds ===


,metric,mean_percent,std_percent,mean ± std
0,R@1,77.67,0.81,77.67 ± 0.81
1,R@5,94.16,0.46,94.16 ± 0.46
2,R@10,97.32,0.22,97.32 ± 0.22
3,mAP,40.32,0.81,40.32 ± 0.81


## Summary

In [ ]:
summary = {
    "model":            "CLIP ViT-B/16",
    "input":            "RGB bands from EuroSAT-MS: B04, B03, B02",
    "split":            "train/query(val)/gallery(test) = 80/10/10",
    "protocol":         "image-to-image retrieval — val→query, test→gallery",
    "num_queries":      int(query_result["features"].shape[0]),
    "num_gallery":      int(gallery_result["features"].shape[0]),
    "num_classes":      int(len(dataset.class_names)),
    "R@1_percent":      round(metrics["R@1"],  2),
    "R@5_percent":      round(metrics["R@5"],  2),
    "R@10_percent":     round(metrics["R@10"], 2),
    "mAP_percent":      round(metrics["mAP"],  2),
}
summary


## Visualise sample retrieval results

For a handful of query images, show the image and its top-5 retrieved gallery images.

In [ ]:
NUM_EXAMPLES = 4
TOP_K_SHOW   = 5

# Collect all query batches into a flat list of images and labels
all_query_imgs, all_query_labels_list = [], []
for batch in query_loader:
    all_query_imgs.append(batch["image"])
    all_query_labels_list.extend(batch["label"].tolist())
all_query_imgs = torch.cat(all_query_imgs, dim=0)  # [Q, 13, H, W]

all_gallery_imgs, all_gallery_labels_list = [], []
for batch in gallery_loader:
    all_gallery_imgs.append(batch["image"])
    all_gallery_labels_list.extend(batch["label"].tolist())
all_gallery_imgs = torch.cat(all_gallery_imgs, dim=0)  # [N, 13, H, W]

sample_query_ids = list(range(0, len(all_query_labels_list), max(1, len(all_query_labels_list) // NUM_EXAMPLES)))[:NUM_EXAMPLES]

for q_pos, q_id in enumerate(sample_query_ids):
    q_class = idx_to_class[all_query_labels_list[q_id]]
    top5_gal_idx = ranked_indices[q_id, :TOP_K_SHOW].tolist()

    fig, axes = plt.subplots(1, TOP_K_SHOW + 1, figsize=(3 * (TOP_K_SHOW + 1), 3))
    q_rgb = extract_rgb_bands(all_query_imgs[q_id:q_id+1])[0]
    axes[0].imshow(stretch_for_display(q_rgb))
    axes[0].set_title(f"QUERY\n{q_class}", fontsize=9, color="white",
                     bbox=dict(facecolor="#333", alpha=0.7, pad=2))
    axes[0].axis("off")

    for rank, g_id in enumerate(top5_gal_idx):
        g_class = idx_to_class[all_gallery_labels_list[g_id]]
        score   = float(similarity[q_id, g_id].item())
        correct = (g_class == q_class)
        g_rgb   = extract_rgb_bands(all_gallery_imgs[g_id:g_id+1])[0]
        axes[rank + 1].imshow(stretch_for_display(g_rgb))
        color = "lime" if correct else "red"
        axes[rank + 1].set_title(
            f"Rank {rank+1}\n{g_class}\ncos={score:.3f}",
            fontsize=8, color=color,
            bbox=dict(facecolor="#222", alpha=0.7, pad=2),
        )
        for spine in axes[rank + 1].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
        axes[rank + 1].axis("off")

    plt.suptitle(f"Query image (val split), class: {q_class}", y=1.02, fontsize=10)
    plt.tight_layout()
    plt.show()
